# SimWorld Studio

**AI-powered 3D scene generation platform** — chat with Claude to build urban scenes in real-time.

## What this notebook does
1. Checks your Colab GPU
2. Downloads the minimal SimWorld binary (~15 GB)
3. Installs SimWorld Studio platform
4. Sets up Claude authentication (API key or Claude Code login)
5. Launches everything and gives you a browser URL

**Run all cells in order.** Total setup time: ~5 minutes.

---

## Cell 1: GPU Check

In [1]:
"""Verify GPU is available — T4 or better required."""
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                       capture_output=True, text=True)
if result.returncode != 0:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> GPU (T4)")
    sys.exit(1)

gpu_info = result.stdout.strip()
print(f"GPU: {gpu_info}")

supported = ["T4", "A100", "V100", "L4", "A10"]
if not any(g in gpu_info for g in supported):
    print(f"WARNING: Unrecognized GPU. Supported: {supported}")
    print("Continuing anyway...")
else:
    print("GPU check passed!")

GPU: Tesla T4, 15360 MiB
GPU check passed!


## Cell 2: Download SimWorld (Minimal Binary)

Downloads a minimal UE Editor binary (~15 GB compressed) with just the assets needed for the Studio demo.

In [2]:
"""Download and extract the minimal SimWorld binary."""
import os

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
SIMWORLD_ARCHIVE = "/tmp/SimWorld-Studio-Minimal.tar.gz"
DOWNLOAD_URL = "https://huggingface.co/datasets/SimWorld-AI/SimWorld-Studio/resolve/main/SimWorld-Studio-Minimal.tar.gz"

UE_BINARY = os.path.join(SIMWORLD_DIR, "Engine", "Binaries", "Linux", "UnrealEditor")

if os.path.exists(UE_BINARY):
    print(f"SimWorld binary already installed at: {SIMWORLD_DIR}")
else:
    print("Downloading SimWorld minimal binary (~15 GB)...")
    !wget -q --show-progress -O {SIMWORLD_ARCHIVE} {DOWNLOAD_URL}
    print("Extracting...")
    !tar xzf {SIMWORLD_ARCHIVE} -C /content/
    !rm -f {SIMWORLD_ARCHIVE}

    if os.path.exists(UE_BINARY):
        os.chmod(UE_BINARY, 0o755)
        # Also chmod the launch script
        launch_sh = os.path.join(SIMWORLD_DIR, "SimWorld-Studio.sh")
        if os.path.exists(launch_sh):
            os.chmod(launch_sh, 0o755)
        print(f"SimWorld binary installed at: {SIMWORLD_DIR}")
    else:
        print("ERROR: UnrealEditor not found after extraction!")
        !find {SIMWORLD_DIR} -maxdepth 3 -type f -name "UnrealEditor*" | head -5

print(f"\nUE Binary: {UE_BINARY}")
print(f"Project: {os.path.join(SIMWORLD_DIR, 'gym_citynav', 'gym_citynav.uproject')}")

SimWorld binary already installed at: /content/SimWorld-Studio-Minimal

UE Binary: /content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor
Project: /content/SimWorld-Studio-Minimal/gym_citynav/gym_citynav.uproject


## Cell 3: Install Studio Platform

In [3]:
"""Install the Studio platform + Claude Code CLI."""
import subprocess, shutil, os

# --- Studio platform (pip package from GitHub) ---
print("[1/3] Installing SimWorld Studio...")

STUDIO_GIT_URL = "git+https://github.com/SimWorld-AI/SimWorld-Studio.git#subdirectory=packaging"

result = subprocess.run(
    ["pip", "install", "-q", STUDIO_GIT_URL],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("  Installed from GitHub.")
else:
    print(f"  ERROR: Could not install package.")
    print(f"  {result.stderr[:200]}")
    print("  Please check the GitHub URL or install manually.")

# Verify installation
studio_installed = shutil.which("simworld-studio") is not None
if studio_installed:
    ver = subprocess.run(["simworld-studio", "version"], capture_output=True, text=True).stdout.strip()
    print(f"  {ver}")
else:
    print("  WARNING: simworld-studio CLI not found after install!")

# --- Node.js (usually pre-installed in Colab) ---
print("[2/3] Checking Node.js...")
if not shutil.which("node"):
    print("  Installing Node.js...")
    !apt-get install -y -qq nodejs npm
else:
    node_ver = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Node.js {node_ver} found.")

# --- Claude Code CLI ---
print("[3/3] Installing Claude Code CLI...")
if not shutil.which("claude"):
    !npm install -g @anthropic-ai/claude-code 2>/dev/null | tail -2
    print("  Claude Code CLI installed.")
else:
    claude_ver = subprocess.run(["claude", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"  Claude Code CLI {claude_ver} already installed.")

print("\nAll dependencies installed!")

[1/3] Installing SimWorld Studio...
  Installed from GitHub.
  simworld-studio v0.2.0
[2/3] Checking Node.js...
  Node.js v20.19.0 found.
[3/3] Installing Claude Code CLI...
  Claude Code CLI 2.1.90 (Claude Code) already installed.

All dependencies installed!


## Cell 4: Authenticate with Claude

You have **two options** to authenticate:

**Option A (recommended):** Run the cell below and enter your Anthropic API key. Get one at [console.anthropic.com](https://console.anthropic.com).

**Option B:** If you have a Claude account with Claude Code access, you can run `!claude login` in a new cell instead. This uses OAuth — no API key needed.

Your credentials stay **entirely local** in this Colab runtime and are never sent to SimWorld servers.

In [4]:
"""Authenticate with Claude — API key or Claude Code login."""
import subprocess, os

# Check if already authenticated via claude login (OAuth)
claude_auth = subprocess.run(["claude", "auth", "status"], capture_output=True, text=True)
already_logged_in = claude_auth.returncode == 0 and "authenticated" in claude_auth.stdout.lower()

if already_logged_in:
    print("Already authenticated via Claude Code login!")
    print("(To switch to API key, set ANTHROPIC_API_KEY below)")
elif os.environ.get("ANTHROPIC_API_KEY"):
    print(f"API key already set (starts with {os.environ['ANTHROPIC_API_KEY'][:10]}...)")
else:
    print("Choose authentication method:\n")
    print("  Option A: Enter API key below")
    print("  Option B: Run '!claude login' in a new cell\n")

    from getpass import getpass
    api_key = getpass("Enter your Anthropic API key (sk-ant-...), or press Enter to skip: ")

    if api_key.strip():
        os.environ["ANTHROPIC_API_KEY"] = api_key.strip()
        if not api_key.startswith("sk-ant-"):
            print("WARNING: Key doesn't start with 'sk-ant-'. Double-check your key.")
            print("Get yours at: https://console.anthropic.com")
        else:
            print("API key set! (stored only in this runtime's memory)")
    else:
        print("No API key entered. Use '!claude login' in a new cell to authenticate via OAuth.")

Choose authentication method:

  Option A: Enter API key below
  Option B: Run '!claude login' in a new cell

Enter your Anthropic API key (sk-ant-...), or press Enter to skip: ··········
API key set! (stored only in this runtime's memory)


## Cell 5: Launch SimWorld + Studio

In [5]:
"""Start SimWorld (headless GPU) and the Studio backend — non-root fix for Colab."""

import subprocess, time, os, socket, pwd

print("[1/5] Installing rendering dependencies...")
r = subprocess.run(
    ["apt-get", "install", "-y", "-qq",
     "xvfb", "vulkan-tools", "mesa-vulkan-drivers",
     "libvulkan1", "libegl1", "libgles2", "libgl1"],
    capture_output=True
)
if r.returncode != 0:
    print(f"  WARNING: Some rendering deps may have failed (exit {r.returncode})")
else:
    print("  Rendering deps installed.")

print("[2/5] Creating non-root user for Unreal...")
USERNAME = "simuser"
HOME_DIR = f"/home/{USERNAME}"

# Create user only if it doesn't already exist
try:
    pwd.getpwnam(USERNAME)
    print(f"  User '{USERNAME}' already exists.")
except KeyError:
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", USERNAME], check=True)
    print(f"  Created user '{USERNAME}'.")

SIMWORLD_DIR = "/content/SimWorld-Studio-Minimal"
WORKSPACE_DIR = "/content/studio_workspace"

os.makedirs(WORKSPACE_DIR, exist_ok=True)

# Give the non-root user ownership of relevant dirs
subprocess.run(["chown", "-R", f"{USERNAME}:{USERNAME}", SIMWORLD_DIR], check=True)
subprocess.run(["chown", "-R", f"{USERNAME}:{USERNAME}", WORKSPACE_DIR], check=True)

# Also make sure home exists / is writable
os.makedirs(HOME_DIR, exist_ok=True)
subprocess.run(["chown", "-R", f"{USERNAME}:{USERNAME}", HOME_DIR], check=True)

print("[3/5] Starting virtual display...")
subprocess.run(["pkill", "-f", "Xvfb"], capture_output=True)
time.sleep(1)

xvfb_proc = subprocess.Popen(
    ["Xvfb", ":99", "-screen", "0", "1280x720x24"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)
print("  Virtual display :99 started.")

print("[4/5] Launching SimWorld as non-root user...")

UE_BINARY = os.path.join(SIMWORLD_DIR, "Engine", "Binaries", "Linux", "UnrealEditor")
PROJECT_FILE = os.path.join(SIMWORLD_DIR, "gym_citynav", "gym_citynav.uproject")
MCP_PORT = 55559

ue_cmd = f"""
export DISPLAY=:99
export HOME={HOME_DIR}
cd {SIMWORLD_DIR}
"{UE_BINARY}" "{PROJECT_FILE}" /Game/Maps/Empty.umap \
  -MCPPort={MCP_PORT} \
  -RenderOffScreen -Unattended \
  -NOSPLASH -NOSOUND \
  -ResX=1280 -ResY=720 \
  -FPSMAX=15 \
  -Messaging
"""

ue_proc = subprocess.Popen(
    ["runuser", "-u", USERNAME, "--", "bash", "-lc", ue_cmd],
    stdout=open("/content/ue.log", "w"),
    stderr=subprocess.STDOUT,
)

print(f"  Waiting for MCP port {MCP_PORT}...")
started = False
for attempt in range(90):  # up to ~3 min
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(2)
        s.connect(("127.0.0.1", MCP_PORT))
        s.close()
        started = True
        break
    except (ConnectionRefusedError, socket.timeout, OSError):
        time.sleep(2)
        if attempt % 10 == 0 and attempt > 0:
            print(f"  Still waiting... ({attempt*2}s)")

if started:
    print("  SimWorld is running!")
else:
    print("  ERROR: SimWorld did not start.")
    print("  Last 30 lines of /content/ue.log:")
    subprocess.run(["tail", "-30", "/content/ue.log"])

print("[5/5] Starting Studio backend as non-root user...")

studio_cmd = f"""
export DISPLAY=:99
export HOME={HOME_DIR}
export ANTHROPIC_API_KEY="{os.environ.get('ANTHROPIC_API_KEY', '')}"
cd {SIMWORLD_DIR}
simworld-studio start \
  --ue-host 127.0.0.1 \
  --ue-port {MCP_PORT} \
  --port 3002 \
  --data-dir {WORKSPACE_DIR}
"""

studio_proc = subprocess.Popen(
    ["runuser", "-u", USERNAME, "--", "bash", "-lc", studio_cmd],
    stdout=open("/content/studio.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(10)

import requests
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    print(f"  Studio backend running! Health: {r.json()}")
except Exception as e:
    print(f"  Studio backend not up yet: {e}")
    print("  Last 30 lines of /content/studio.log:")
    subprocess.run(["tail", "-30", "/content/studio.log"])

print("\nLaunch sequence complete.")

[1/4] Installing rendering dependencies...
  Rendering deps installed.
[2/4] Starting virtual display...
  Virtual display :99 started.
[3/4] Launching SimWorld (this takes 30-60 seconds)...
  Waiting for MCP port 55559...
  Still waiting... (20s)
  Still waiting... (40s)
  Still waiting... (60s)
  Still waiting... (80s)
  Still waiting... (100s)
  Check /content/ue.log for details.
  Last 10 lines of log:
Refusing to run with the root privileges.
libc++abi: __cxa_guard_acquire detected recursive initialization
[4/4] Starting Studio backend...
  Studio backend may still be starting: HTTPConnectionPool(host='localhost', port=3002): Max retries exceeded with url: /api/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e6d4b4d0710>: Failed to establish a new connection: [Errno 111] Connection refused'))
  Check: !cat /content/studio.log

All services launched!


## Cell 6: Get Your Browser URL

In [6]:
"""Create a public tunnel URL to access the Studio in your browser."""
import subprocess, time, re, os, select

# Install cloudflared
print("Setting up tunnel...")
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(
        ["wget", "-q",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"],
        check=True
    )
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# Kill any existing tunnel
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Start tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3002", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for URL — use select() for non-blocking reads
public_url = None
accumulated = ""
for _ in range(30):  # wait up to 30 seconds
    time.sleep(1)
    try:
        ready, _, _ = select.select([tunnel.stderr], [], [], 0)
        if ready:
            chunk = os.read(tunnel.stderr.fileno(), 8192).decode(errors='replace')
            accumulated += chunk
            match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', accumulated)
            if match:
                public_url = match.group(0)
                break
    except Exception:
        pass

if public_url:
    print(f"")
    print(f"{'='*60}")
    print(f"  SimWorld Studio is live!")
    print(f"")
    print(f"  Open in browser: {public_url}")
    print(f"{'='*60}")
    print(f"")
    print(f"Chat with Claude to build 3D urban scenes!")
    print(f"Try: 'Build a small neighborhood with 4 houses and trees'")
else:
    print("Tunnel failed to start. Trying Colab proxy fallback...")
    try:
        from google.colab.output import eval_js
        proxy_url = eval_js("google.colab.kernel.proxyPort(3002)")
        public_url = proxy_url
        print(f"Open in browser: {proxy_url}")
    except Exception:
        print("Both tunnel methods failed. Check your connection.")
        public_url = "http://localhost:3002"

Setting up tunnel...

  SimWorld Studio is live!

  Open in browser: https://concern-reproduced-sanyo-trance.trycloudflare.com

Chat with Claude to build 3D urban scenes!
Try: 'Build a small neighborhood with 4 houses and trees'


## Cell 7: Verify Everything Works

Run this cell to confirm all services are healthy before using the Studio.

In [7]:
"""Automated verification — all checks must pass before using the Studio."""
import requests, socket, subprocess, os

print("Running verification checks...\n")

results = {}

# 1. GPU
r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                   capture_output=True, text=True)
results["GPU"] = (r.returncode == 0, r.stdout.strip() if r.returncode == 0 else "Not found")

# 2. SimWorld binary
ue_bin = "/content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor"
results["SimWorld Binary"] = (os.path.exists(ue_bin), ue_bin if os.path.exists(ue_bin) else "Not found")

# 3. SimWorld TCP (MCP port)
mcp_port = globals().get('MCP_PORT', 55559)
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect(("127.0.0.1", mcp_port))
    s.close()
    results[f"SimWorld MCP ({mcp_port})"] = (True, "Connected")
except Exception as e:
    results[f"SimWorld MCP ({mcp_port})"] = (False, str(e))

# 4. Studio Backend
try:
    r = requests.get("http://localhost:3002/api/health", timeout=10)
    results["Studio Backend"] = (r.status_code == 200, f"HTTP {r.status_code}")
except Exception as e:
    results["Studio Backend"] = (False, str(e))

# 5. Skills
try:
    r = requests.get("http://localhost:3002/api/skills", timeout=10)
    data = r.json()
    count = len(data) if isinstance(data, list) else 0
    results["Skills"] = (count > 0, f"{count} skills loaded")
except Exception as e:
    results["Skills"] = (False, str(e))

# 6. Assets
try:
    r = requests.get("http://localhost:3002/api/assets", timeout=10)
    results["Assets"] = (r.status_code == 200, "Catalog loaded")
except Exception as e:
    results["Assets"] = (False, str(e))

# 7. Claude Code CLI
r = subprocess.run(["claude", "--version"], capture_output=True, text=True)
results["Claude Code CLI"] = (r.returncode == 0,
    r.stdout.strip() if r.returncode == 0 else "Not installed")

# 8. Authentication (API key OR Claude Code login)
key_set = bool(os.environ.get("ANTHROPIC_API_KEY", ""))
claude_auth = subprocess.run(["claude", "auth", "status"], capture_output=True, text=True)
oauth_ok = claude_auth.returncode == 0 and "authenticated" in claude_auth.stdout.lower()
auth_ok = key_set or oauth_ok
auth_detail = "API key" if key_set else ("Claude Code login" if oauth_ok else "NOT SET — run Cell 4")
results["Authentication"] = (auth_ok, auth_detail)

# 9. Tunnel
tunnel_url = globals().get('public_url', '')
tunnel_ok = bool(tunnel_url) and tunnel_url.startswith("http")
results["Public URL"] = (tunnel_ok, tunnel_url if tunnel_ok else "Not created")

# Print results
all_ok = True
for name, (ok, detail) in results.items():
    icon = "[PASS]" if ok else "[FAIL]"
    if not ok:
        all_ok = False
    print(f"  {icon} {name}: {detail}")

print()
if all_ok:
    print("ALL CHECKS PASSED! SimWorld Studio is ready.")
    if tunnel_url:
        print(f"Open {tunnel_url} in your browser to start building 3D scenes.")
else:
    print("SOME CHECKS FAILED. Common fixes:")
    print("  - SimWorld MCP: wait 30-60s more, then re-run this cell")
    print("  - Authentication: run Cell 4 or use '!claude login'")
    print("  - Tunnel: re-run Cell 6")
    print("  - Studio Backend: !cat /content/studio.log")

Running verification checks...

  [PASS] GPU: Tesla T4
  [PASS] SimWorld Binary: /content/SimWorld-Studio-Minimal/Engine/Binaries/Linux/UnrealEditor
  [FAIL] SimWorld MCP (55559): [Errno 111] Connection refused
  [FAIL] Studio Backend: HTTPConnectionPool(host='localhost', port=3002): Max retries exceeded with url: /api/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e6d4b63c500>: Failed to establish a new connection: [Errno 111] Connection refused'))
  [FAIL] Skills: HTTPConnectionPool(host='localhost', port=3002): Max retries exceeded with url: /api/skills (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e6d4b4feea0>: Failed to establish a new connection: [Errno 111] Connection refused'))
  [FAIL] Assets: HTTPConnectionPool(host='localhost', port=3002): Max retries exceeded with url: /api/assets (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e6d4b4fca70>: Failed to establish a new conne

## Cell 8: Smoke Test (End-to-End)

Sends a test prompt through the full pipeline: **Chat -> Claude -> MCP Tools -> SimWorld**

In [8]:
"""End-to-end smoke test — verifies the full pipeline works."""
import requests, json

print("Sending test prompt through the full pipeline...\n")
print("Prompt: 'Set up the environment with a sunny sky, then spawn one small building'\n")

try:
    response = requests.post("http://localhost:3002/api/chat", json={
        "message": "Set up the environment with a sunny sky, then spawn one small residential building (BP_Building_01) at the origin. Then take a screenshot.",
        "sessionId": None,
        "skills": ["building_placement"]
    }, stream=True, timeout=180)

    tool_calls = []
    text_output = []
    screenshots = []
    errors = []
    current_event = None  # Track SSE event type

    for line in response.iter_lines():
        if not line:
            continue
        decoded = line.decode('utf-8')

        # SSE comment lines (heartbeat)
        if decoded.startswith(':'):
            continue

        # Parse SSE event/data pairs
        if decoded.startswith('event: '):
            current_event = decoded[7:]
            continue
        if not decoded.startswith('data: '):
            continue

        try:
            data = json.loads(decoded[6:])
        except json.JSONDecodeError:
            continue

        event_type = current_event or 'unknown'

        if event_type == 'text':
            delta = data.get('delta', '')
            text_output.append(delta)
            print(delta, end='', flush=True)
        elif event_type == 'tool_start':
            name = data.get('displayName', data.get('name', '?'))
            tool_calls.append(name)
            print(f"\n  >> Tool: {name}", flush=True)
        elif event_type == 'tool_result':
            result = data.get('result', '')[:200]
            is_error = data.get('isError', False)
            if is_error:
                errors.append(result)
                print(f"\n  >> ERROR: {result}", flush=True)
            else:
                print(f"\n  >> Result: {result[:100]}...", flush=True)
        elif event_type == 'screenshot':
            screenshots.append(data.get('filepath', ''))
            print(f"\n  >> Screenshot captured!", flush=True)
        elif event_type == 'done':
            cost = data.get('costUsd')
            if cost:
                print(f"\n\n  Cost: ${cost:.4f}")

    print(f"\n\n{'='*50}")
    print(f"Smoke Test Results:")
    print(f"  Tool calls:  {len(tool_calls)} ({', '.join(tool_calls)})")
    print(f"  Screenshots: {len(screenshots)}")
    print(f"  Errors:      {len(errors)}")

    if len(tool_calls) > 0 and len(errors) == 0:
        print(f"\n  SMOKE TEST PASSED!")
        print(f"  Full pipeline working: Chat -> Claude -> MCP -> SimWorld")
    elif len(tool_calls) > 0:
        print(f"\n  PARTIAL PASS — tools fired but some errors occurred.")
    else:
        print(f"\n  SMOKE TEST FAILED — no tool calls detected.")
        print(f"  Check: API key set? SimWorld running? Logs: /content/studio.log")

except requests.exceptions.Timeout:
    print("\nSmoke test timed out (180s). Claude may be slow to respond.")
    print("The platform may still work — try using the browser UI.")
except Exception as e:
    print(f"\nSmoke test error: {e}")
    print("Check /content/studio.log for backend errors.")

Sending test prompt through the full pipeline...

Prompt: 'Set up the environment with a sunny sky, then spawn one small building'


Smoke test error: HTTPConnectionPool(host='localhost', port=3002): Max retries exceeded with url: /api/chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7e6d60418260>: Failed to establish a new connection: [Errno 111] Connection refused'))
Check /content/studio.log for backend errors.


---

## Troubleshooting

| Issue | Fix |
|---|---|
| "No GPU detected" | Runtime -> Change runtime type -> GPU (T4) |
| SimWorld TCP fails | Wait 60s more, re-run Cell 7 |
| Studio backend fails | Check: `!cat /content/studio.log` |
| Tunnel fails | Re-run Cell 6, or use Colab proxy |
| Claude errors | Verify API key in Cell 4 |
| Session expires | Colab free tier = 12h max. Re-run all cells. |

### View Logs
```python
!tail -50 /content/ue.log      # SimWorld logs
!tail -50 /content/studio.log    # Studio backend logs
```